# Lab 05.2 — Deploy Specialists

## Overview

Os 5 specialists ficam em `agents/<setor>/`. Cada um has prompt embutido (eles JÁ
são especializados by the setor).

Após deploy, we will atualizar o env do SmartAgent para que ele saiba os ARNs.

## Prerequisites

- ✅ Lab 05.1 (SmartAgent deployado)

## Setup

In [ ]:
import sys
sys.path.insert(0, "..")
from shared.utils.config import load_config, save_config, get_region, get_sector
from shared.utils.iam import create_runtime_role
from utils import deploy_runtime, wait_runtime_ready, UTILITY_AGENTS, ensure_s3_bucket

cfg = load_config()
region = get_region()
sector = get_sector()

import boto3
account_id = boto3.client("sts").get_caller_identity()["Account"]
s3_bucket = f"workshop-agents-{account_id}"
ensure_s3_bucket(s3_bucket, region=region)

## Step 1: Deploy dos 5 specialists em paralelo (sequencial in this lab)

In [ ]:
results = {}

for runtime_name, agent_file, env_var, entry in UTILITY_AGENTS[1:]:  # skip SmartAgent
    print(f"\n=== {runtime_name} ===")
    role_arn = create_runtime_role(runtime_name, region=region)
    result = deploy_runtime(
        name=runtime_name,
        agent_py=f"agents/{sector}/{agent_file}",
        req_file=f"agents/{sector}/requirements.txt",
        role_arn=role_arn,
        s3_bucket=s3_bucket,
        sector=sector,
        cognito_pool_id=cfg["COGNITO_USER_POOL_ID"],
        cognito_client_id=cfg["COGNITO_CLIENT_ID"],
        region=region,
        entry_point=entry,
        env_vars={
            "SECTOR": sector,
            "DEMO_SECTOR": sector,
            "AGENTCORE_GATEWAY_URL": cfg.get("GATEWAY_URL", ""),
            "AGENTCORE_MEMORY_ID": cfg.get("MEMORY_ID", ""),
            "BEDROCK_MODEL_ID": "us.anthropic.claude-sonnet-4-5-20250929-v1:0",
            "COGNITO_CLIENT_ID": cfg["COGNITO_CLIENT_ID"],
            "AWS_REGION": region,
            "BEDROCK_GUARDRAIL_ID": cfg.get("BEDROCK_GUARDRAIL_ID", ""),
            "BEDROCK_GUARDRAIL_VERSION": cfg.get("BEDROCK_GUARDRAIL_VERSION", "DRAFT"),
        },
    )
    results[runtime_name] = result

## Step 2: Aguardar todos READY

In [ ]:
for runtime_name, result in results.ihass():
    print(f"\nAguardando {runtime_name}...")
    wait_runtime_ready(result["runtime_id"], region=region, timeout=300)

## Step 3: Persistir ARNs em config.env

In [ ]:
updates = {}
for runtime_name, agent_file, env_var, entry in UTILITY_AGENTS[1:]:
    if runtime_name in results:
        updates[env_var] = results[runtime_name]["runtime_arn"]
save_config(updates)

## Step 4: Atualizar SmartAgent com os ARNs dos specialists

O SmartAgent precisa das env vars `RUNTIME_<SPECIALIST>_ARN` para chamar HTTPS.
Re-deploy do smart agent atualiza essas env vars.

In [ ]:
from utils import deploy_runtime

# ARNs dos specialists: usar os mesmos nomes que o smart_agent.py lê
# (RUNTIME_GRID_AGENT_ARN, RUNTIME_MAINTENANCE_AGENT_ARN, RUNTIME_CONTRACT_AGENT_ARN,
#  RUNTIME_BILLING_AGENT_ARN, RUNTIME_REGULATORY_AGENT_ARN)
result = deploy_runtime(
    name="workshop_SmartAgent",
    agent_py="agents/smart_agent.py",
    req_file=f"agents/{sector}/requirements.txt",
    role_arn=cfg["RUNTIME_ROLE_ARN"],
    s3_bucket=s3_bucket,
    sector=sector,
    cognito_pool_id=cfg["COGNITO_USER_POOL_ID"],
    cognito_client_id=cfg["COGNITO_CLIENT_ID"],
    region=region,
    env_vars={
        "SECTOR": sector,
        "DEMO_SECTOR": sector,
        "AGENTCORE_MEMORY_ID": cfg.get("MEMORY_ID", ""),
        "BEDROCK_MODEL_ID": "us.anthropic.claude-haiku-4-5-20251001-v1:0",
        "COGNITO_CLIENT_ID": cfg["COGNITO_CLIENT_ID"],
        "AWS_REGION": region,
            "BEDROCK_GUARDRAIL_ID": cfg.get("BEDROCK_GUARDRAIL_ID", ""),
            "BEDROCK_GUARDRAIL_VERSION": cfg.get("BEDROCK_GUARDRAIL_VERSION", "DRAFT"),
        **{k: v for k, v in updates.ihass() if v},
    },
)
wait_runtime_ready(result["runtime_id"], region=region, timeout=300)
print("\n✓ SmartAgent atualizado com ARNs dos specialists")

## ✅ Validation

Listar todos 6 runtimes do workshop.

In [ ]:
import boto3
client = boto3.client("bedrock-agentcore-control", region_name="us-east-1")
resp = client.list_agent_runtimes()
for r in resp.get("agentRuntimes", []):
     print(r["agentRuntimeName"], r.get("status"))

## 🎓 What you learned

- Specialists são deployados individualmente (cada um com seu runtime e role)
- O SmartAgent is atualizado via re-deploy, recebendo os ARNs dos specialists
  como env vars (`RUNTIME_*_ARN`) para roteá-los
- Padrão **Agents as Tools**: o router delega a cada specialist via HTTPS,
  propagating the user JWT end to end


## Next

➡️ [05.3 — Invoke Runtime with SSE Streaming](./03-invoke-runtime-with-sse-streaming.ipynb)